Install the libraries

In [1]:
!pip install scikit-learn
!pip install torch


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

#Load Dataset
data = np.load("processed_eye_data.npz")
X = data["X"]
y = data["y"]
fs = 250  # Sampling frequency

# Check the shapes of the loaded data
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (107, 168)
y shape: (107,)


In [3]:
# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [4]:
import joblib
joblib.dump(scaler, "feature_scaler.pkl")

['feature_scaler.pkl']

Training The Neural Network, Set Up Data

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Convert Labels to 0 / 1
y_train = y_train - 1
y_test  = y_test - 1

#Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

#Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Model Architecture

In [6]:
class EyeClassifier(nn.Module):

    def __init__ (self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)  # Output layer for binary classification
        )

    def forward(self, x):
        return self.net(x)

In [7]:
peak_all_time_test_acc = 0.0

Training Loop Log Loss every epoch for train and test set

In [8]:
model = EyeClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100

peak_test_acc = 0.0
peak_test_acc_epoch = 0
record_broken = False

for epoch in range(num_epochs):
    # ---------------- TRAIN ----------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)              # logits
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += y_batch.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # ---------------- TEST ----------------
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == y_batch).sum().item()
            test_total += y_batch.size(0)

    test_loss /= len(test_loader)
    test_acc = test_correct / test_total

    if test_acc > peak_test_acc:
        peak_test_acc = test_acc
        peak_test_acc_epoch = epoch + 1

    if test_acc > peak_all_time_test_acc:
        peak_all_time_test_acc = test_acc
        torch.save(model.state_dict(), "eye_model.pth")
        print(f"New best model saved with test accuracy: {peak_all_time_test_acc*100:.2f}% at epoch {epoch+1}")
        record_broken = True

    # ---------------- LOG ----------------
    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc*100:.2f}%"
    )
print(f"Peak Test Accuracy so far: {peak_test_acc*100:.2f}")
print(f"Peak Test Accuracy Epoch: {peak_test_acc_epoch}")

if record_broken:
    print(f"New record achieved with test accuracy: {peak_all_time_test_acc*100:.2f}% at epoch {peak_test_acc_epoch}")

New best model saved with test accuracy: 63.64% at epoch 1
Epoch 01/100 | Train Loss: 0.7191, Train Acc: 43.53% | Test Loss: 0.6688, Test Acc: 63.64%
Epoch 02/100 | Train Loss: 0.7106, Train Acc: 43.53% | Test Loss: 0.6697, Test Acc: 63.64%
Epoch 03/100 | Train Loss: 0.7051, Train Acc: 43.53% | Test Loss: 0.6704, Test Acc: 63.64%
Epoch 04/100 | Train Loss: 0.6993, Train Acc: 44.71% | Test Loss: 0.6720, Test Acc: 63.64%
Epoch 05/100 | Train Loss: 0.6879, Train Acc: 44.71% | Test Loss: 0.6734, Test Acc: 63.64%
Epoch 06/100 | Train Loss: 0.6802, Train Acc: 49.41% | Test Loss: 0.6746, Test Acc: 63.64%
Epoch 07/100 | Train Loss: 0.6630, Train Acc: 58.82% | Test Loss: 0.6787, Test Acc: 59.09%
Epoch 08/100 | Train Loss: 0.6470, Train Acc: 76.47% | Test Loss: 0.6830, Test Acc: 59.09%
Epoch 09/100 | Train Loss: 0.6300, Train Acc: 82.35% | Test Loss: 0.6900, Test Acc: 45.45%
Epoch 10/100 | Train Loss: 0.6096, Train Acc: 81.18% | Test Loss: 0.6931, Test Acc: 50.00%
Epoch 11/100 | Train Loss: 0.58

In [9]:
peak_all_time_test_acc

0.7727272727272727

Metrics

In [10]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)              # logits
        preds = torch.argmax(outputs, dim=1) # class indices (0/1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())

# Concatenate batches
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

In [11]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[8 6]
 [2 6]]


In [12]:
print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Eyes Open", "Eyes Closed"]
))

Classification Report:
              precision    recall  f1-score   support

   Eyes Open       0.80      0.57      0.67        14
 Eyes Closed       0.50      0.75      0.60         8

    accuracy                           0.64        22
   macro avg       0.65      0.66      0.63        22
weighted avg       0.69      0.64      0.64        22

